# 🧠 Classification with Keras — MNIST Multi-Class Recognition

## 📋 Overview

In this notebook, I build a **multi-class image classification neural network** using Keras to recognise handwritten digits from the MNIST dataset — a foundational benchmark in deep learning.

The workflow covers the full pipeline:
1. 📥 Load and explore the MNIST dataset (60k train / 10k test images)
2. ⚙️ Preprocess: flatten, normalise, and one-hot encode
3. 🏗️ Build a fully-connected `Sequential` neural network with `Dense` layers
4. 🔄 Train with Adam optimiser and categorical cross-entropy loss
5. 📊 Evaluate accuracy on the held-out test set
6. 💾 Save and reload the trained model
7. 🧪 Experiment with depth — compare 3-layer vs 6-layer architectures

> 🔌 **Telecom analogy:** MNIST classification is structurally identical to modulation recognition in wireless systems. In both cases, the input is a signal representation (pixel grid vs. IQ samples), and the output is one of $C$ discrete classes (digit 0–9 vs. modulation type QPSK, 64-QAM…). The same Softmax + cross-entropy machinery applies.


## 🧩 Theory

### Multi-Class Classification

For a problem with $C$ classes, the network assigns a probability to each class. The **Softmax** function at the output layer converts raw scores (logits) $z_c$ into a valid probability distribution:

$$\hat{y}_c = \text{Softmax}(z_c) = \frac{e^{z_c}}{\sum_{j=1}^{C} e^{z_j}}, \quad \sum_{c=1}^{C} \hat{y}_c = 1$$

### Loss Function — Categorical Cross-Entropy

Training minimises the categorical cross-entropy between the true one-hot label $y$ and the predicted distribution $\hat{y}$:

$$\mathcal{L} = -\sum_{c=1}^{C} y_c \log(\hat{y}_c)$$

When $y$ is a one-hot vector (only one $y_c = 1$, the rest 0), this simplifies to:

$$\mathcal{L} = -\log(\hat{y}_{\text{true class}})$$

The loss is 0 when the model is perfectly confident about the correct class, and grows toward $\infty$ as the predicted probability for the true class approaches 0.

### Network Architecture

| Layer | Neurons | Activation | Role |
|---|---|---|---|
| Input | 784 | — | Flattened 28×28 image |
| Hidden 1 | 784 | ReLU | Feature extraction |
| Hidden 2 | 100 | ReLU | Compression / abstraction |
| Output | 10 | Softmax | Class probabilities |

### ReLU Activation

$$\text{ReLU}(z) = \max(0, z)$$

ReLU is the standard hidden-layer activation. It's computationally cheap and avoids the **vanishing gradient** problem that plagued sigmoid/tanh in deep networks.

### Adam Optimiser

Adam (Adaptive Moment Estimation) adapts the learning rate per parameter using estimates of first and second moments of the gradients:

$$m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t \quad \text{(momentum)}$$
$$v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2 \quad \text{(RMS scaling)}$$
$$\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{\hat{v}_t} + \epsilon} \hat{m}_t$$

Default values: $\beta_1 = 0.9$, $\beta_2 = 0.999$, $\eta = 0.001$.


## ⚙️ Environment Setup

I suppress TensorFlow's CPU-related warnings to keep training output clean. On a GPU build, these lines can be removed.

In [ ]:
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

## Part 1 — 📥 Data Preparation

### Imports

I import Keras utilities and Matplotlib for visualisation. `to_categorical` will handle the one-hot encoding of class labels.


In [ ]:
import keras
import matplotlib.pyplot as plt

from keras.models import Sequential
from keras.layers import Dense, Input
from keras.utils import to_categorical

### 🔢 Loading MNIST

Keras bundles the MNIST dataset and returns it pre-split into training and test sets. 60,000 images for training, 10,000 for testing — each 28×28 grayscale pixels.


In [ ]:
from keras.datasets import mnist

(X_train, y_train), (X_test, y_test) = mnist.load_data()

print(f"Training set:  {X_train.shape}  — {X_train.shape[0]:,} images of {X_train.shape[1]}×{X_train.shape[2]} pixels")
print(f"Test set:      {X_test.shape}  — {X_test.shape[0]:,} images")

### 👁️ Visualising a Sample

Let me confirm what the data looks like before any preprocessing.


In [ ]:
plt.figure(figsize=(4, 4))
plt.imshow(X_train[0], cmap='gray')
plt.title(f"Label: {y_train[0]}", fontsize=14)
plt.axis('off')
plt.tight_layout()
plt.show()
print(f"Pixel range before normalisation: [{X_train.min()}, {X_train.max()}]")

### ⚙️ Preprocessing — Three Steps

A conventional fully-connected neural network expects a **1D input vector**, not a 2D image. I apply three preprocessing operations:

**1. Flatten:** reshape each 28×28 image into a 784-element vector.

$$x_{\text{flat}} \in \mathbb{R}^{784} \quad \text{(from } \mathbb{R}^{28 \times 28}\text{)}$$

Think of it like serialising a 2D spectrum scan into a 1D sample stream — the spatial structure is temporarily discarded in favour of a flat representation the dense layers can process.

**2. Normalise:** scale pixel values from $[0, 255]$ to $[0, 1]$.

$$x_{\text{norm}} = \frac{x_{\text{raw}}}{255}$$

This keeps all input features on the same scale, which stabilises gradient updates during training.

**3. One-hot encode labels:** convert integer class labels (0–9) into binary vectors of length 10.

$$y = 5 \rightarrow [0, 0, 0, 0, 0, 1, 0, 0, 0, 0]$$

This is required because categorical cross-entropy expects a probability distribution target, not a single integer.


In [ ]:
# 1. Flatten: 28×28 → 784
num_pixels = X_train.shape[1] * X_train.shape[2]  # 28 * 28 = 784

X_train = X_train.reshape(X_train.shape[0], num_pixels).astype('float32')
X_test  = X_test.reshape(X_test.shape[0],  num_pixels).astype('float32')

# 2. Normalise: [0, 255] → [0, 1]
X_train = X_train / 255
X_test  = X_test  / 255

# 3. One-hot encode labels
y_train = to_categorical(y_train)
y_test  = to_categorical(y_test)

num_classes = y_test.shape[1]

print(f"X_train shape after preprocessing: {X_train.shape}")
print(f"y_train shape after one-hot encoding: {y_train.shape}")
print(f"Number of classes: {num_classes}")
print(f"Example label for first training image: {y_train[0]}")

## Part 2 — 🏗️ Building the Neural Network

### Architecture

I define the network as a Keras `Sequential` model — a linear stack of layers. Each `Dense` layer applies a linear transformation followed by an activation function:

$$z = W x + b, \quad a = \text{activation}(z)$$

The architecture:

| Layer | Neurons | Activation | Parameters |
|---|---|---|---|
| Input | 784 | — | 0 |
| Dense 1 | 784 | ReLU | 784×784 + 784 = 615,440 |
| Dense 2 | 100 | ReLU | 784×100 + 100 = 78,500 |
| Output | 10 | Softmax | 100×10 + 10 = 1,010 |
| **Total** | | | **~694k parameters** |

The output layer has exactly 10 neurons — one per digit class — with Softmax ensuring the outputs sum to 1 (interpretable as probabilities).


In [ ]:
def classification_model():
    model = Sequential([
        Input(shape=(num_pixels,)),
        Dense(num_pixels, activation='relu'),   # Hidden layer 1 — 784 neurons
        Dense(100, activation='relu'),           # Hidden layer 2 — 100 neurons
        Dense(num_classes, activation='softmax') # Output layer — 10 classes
    ])
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Inspect the architecture
model = classification_model()
model.summary()

## Part 3 — 🔄 Training & Evaluation

### Training

I train for **10 epochs** — meaning the model sees all 60,000 training images 10 times. After each epoch, accuracy on the validation set (X_test, y_test) gives a real-time health check.

> 📡 **Analogy:** This is like running 10 full sweeps of a spectrum analyser across the signal band, adjusting filter coefficients after each pass. Each sweep (epoch) refines the model's understanding of the pattern.

Training progress to watch:
- `loss` and `val_loss` should both decrease
- `accuracy` and `val_accuracy` should both increase
- If `val_loss` starts rising while `train_loss` drops → overfitting


In [ ]:
# Build the model
model = classification_model()

# Train — this takes a few minutes on CPU
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=10,
    verbose=2
)

### 📊 Evaluation

After training, I evaluate on the test set. The model has never seen these 10,000 images during training — this gives an unbiased accuracy estimate.


In [ ]:
scores = model.evaluate(X_test, y_test, verbose=0)
print(f"✅ Test Accuracy : {scores[1]*100:.2f}%")
print(f"❌ Test Error    : {(1 - scores[1])*100:.2f}%")

### 📈 Training Curves

Plotting the accuracy and loss curves over epochs helps diagnose how training went.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Accuracy
axes[0].plot(history.history['accuracy'],     label='Train', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Validation', linewidth=2, linestyle='--')
axes[0].set_title('🎯 Model Accuracy', fontsize=13)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(history.history['loss'],     label='Train', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Validation', linewidth=2, linestyle='--')
axes[1].set_title('📉 Model Loss', fontsize=13)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Categorical Cross-Entropy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Part 4 — 💾 Model Persistence

### Saving the Model

Keras saves models in the `.keras` format — a zipped archive containing:
- **Architecture** (layer structure and configuration)
- **Weights** (trained parameter values)
- **Optimizer state** (so training can resume exactly where it left off)

> 🔧 **Analogy:** This is like checkpointing a long signal processing job — the full state is serialised to disk so the process can be interrupted and resumed without starting over. Standard practice in any production telecom pipeline.


In [ ]:
model.save('classification_model.keras')
print("✅ Model saved to 'classification_model.keras'")

### Loading the Model

I load the saved model using `keras.saving.load_model()`. The loaded model is a complete, ready-to-use object — no need to redefine the architecture or recompile.


In [ ]:
pretrained_model = keras.saving.load_model('classification_model.keras')
print("✅ Pre-trained model loaded successfully")

# Verify it still produces the same accuracy
scores_loaded = pretrained_model.evaluate(X_test, y_test, verbose=0)
print(f"Loaded model accuracy: {scores_loaded[1]*100:.2f}% (should match original)")

## Part 5 — 🧪 Going Deeper: 6-Layer Architecture

### Does depth help here?

I build a 6-layer variant to compare against the 3-layer baseline. The extra 4 hidden layers add more capacity — but also more parameters, slower training, and potential overfitting risk.

**New architecture:**

| Layer | Neurons | Activation |
|---|---|---|
| Input | 784 | — |
| Dense 1 | 784 | ReLU |
| Dense 2 | 100 | ReLU |
| Dense 3 | 100 | ReLU |
| Dense 4 | 100 | ReLU |
| Dense 5 | 100 | ReLU |
| Output | 10 | Softmax |

> 📡 **Analogy:** Adding more layers is like cascading more filter stages in an RF receiver chain. More stages can resolve finer features — but each stage also adds noise and latency. There's a sweet spot.


In [ ]:
def classification_model_6layers():
    model = Sequential([
        Input(shape=(num_pixels,)),
        Dense(num_pixels, activation='relu'),    # Layer 1
        Dense(100, activation='relu'),            # Layer 2
        Dense(100, activation='relu'),            # Layer 3
        Dense(100, activation='relu'),            # Layer 4
        Dense(100, activation='relu'),            # Layer 5
        Dense(num_classes, activation='softmax') # Output
    ])
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Train the 6-layer model
model_6layers = classification_model_6layers()
model_6layers.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=10, verbose=2)

# Evaluate
scores_6layers = model_6layers.evaluate(X_test, y_test, verbose=0)

print(f"\n{'='*45}")
print(f"  3-layer model accuracy : {scores[1]*100:.2f}%")
print(f"  6-layer model accuracy : {scores_6layers[1]*100:.2f}%")
print(f"{'='*45}")

## Part 6 — 🔄 Transfer Learning: Resuming from a Checkpoint

### Load and Continue Training

I reload the earlier saved 3-layer model and train it for **10 more epochs** (total: 20 epochs). This simulates a real-world scenario where training is split across sessions or hardware restarts.

Because I saved the **optimizer state** alongside the weights, training resumes seamlessly — gradients and adaptive learning rates pick up exactly where they left off.


In [ ]:
# Load the previously saved model
pretrained_model = keras.saving.load_model('classification_model.keras')
print("✅ Pre-trained model loaded successfully")

# Continue training for 10 more epochs
pretrained_model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=10,
    verbose=2
)

# Evaluate the extended model
scores_20_epochs = pretrained_model.evaluate(X_test, y_test, verbose=0)

print(f"\n{'='*45}")
print(f"  10 epochs accuracy : {scores[1]*100:.2f}%")
print(f"  20 epochs accuracy : {scores_20_epochs[1]*100:.2f}%")
delta = (scores_20_epochs[1] - scores[1]) * 100
print(f"  Improvement        : {delta:+.2f}%")
print(f"{'='*45}")

## 📊 Summary

### What I built

A fully-connected neural network (MLP) that classifies handwritten digits from the MNIST dataset with ~98% accuracy.

### Pipeline recap

| Step | Operation | Detail |
|---|---|---|
| 📥 Data | Load MNIST | 60k train / 10k test, 28×28 px |
| ⚙️ Flatten | 2D → 1D | 28×28 → 784 vector |
| 🔢 Normalise | Scale pixels | [0, 255] → [0, 1] |
| 🎯 Encode | One-hot labels | Integer → binary vector |
| 🏗️ Architecture | Sequential MLP | Input → 784(ReLU) → 100(ReLU) → 10(Softmax) |
| 📉 Loss | Categorical cross-entropy | $\mathcal{L} = -\sum y_c \log \hat{y}_c$ |
| ⚡ Optimiser | Adam | Adaptive learning rate |
| 🔄 Training | 10 epochs | ~2 min on CPU |
| 💾 Persistence | `.keras` format | Architecture + weights + optimizer state |

### Key results

| Model | Epochs | Test Accuracy |
|---|---|---|
| 3-layer baseline | 10 | ~98% |
| 6-layer deep | 10 | ~98% (marginal gain) |
| 3-layer extended | 20 | ~98.5% (diminishing returns) |

### What I learned

- One-hot encoding transforms the classification problem into a form compatible with Softmax + cross-entropy
- ReLU activations solve the vanishing gradient problem in hidden layers
- Adam handles learning rate scheduling automatically — no manual tuning needed
- More layers don't always help: for MNIST, the 3-layer baseline is near-optimal
- Model persistence is straightforward with `.keras` — full optimizer state is saved

### 🔌 Connection to my background

MNIST digit recognition and **RF modulation classification** share the same mathematical structure:
- Input: signal representation (pixel grid / IQ samples)
- Hidden layers: feature extraction (edges / spectral patterns)
- Output: class probabilities over discrete categories (digits / modulation types)
- Same Softmax + cross-entropy + Adam machinery applies to both

This is a direct entry point into signal intelligence (SIGINT) and spectrum monitoring with deep learning.


## 🧪 Sandbox

This section is for free experimentation. Ideas to try:

1. **Change the hidden layer size** — what happens with 256 neurons instead of 784?
2. **Add Dropout** — regularise the 6-layer model to reduce overfitting
3. **Try SGD instead of Adam** — observe slower convergence
4. **Reduce epochs** — how few epochs are needed to reach 95% accuracy?
5. **Visualise misclassified digits** — which digits get confused most often?


In [ ]:
# 🧪 Sandbox — experiment here

# Example: predict on a few test images and visualise results
predictions = model.predict(X_test[:10])
predicted_classes = predictions.argmax(axis=1)
true_classes = y_test[:10].argmax(axis=1)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_test[i].reshape(28, 28), cmap='gray')
    color = 'green' if predicted_classes[i] == true_classes[i] else 'red'
    ax.set_title(f"Pred: {predicted_classes[i]} | True: {true_classes[i]}", color=color, fontsize=10)
    ax.axis('off')

plt.suptitle("🧪 First 10 Predictions (green = correct, red = wrong)", fontsize=12)
plt.tight_layout()
plt.show()